In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
ratings_schema = StructType(
    [
        StructField("userId", IntegerType(), True),
        StructField("movieId", IntegerType(), True),
        StructField("rating", DoubleType(), True),
        StructField("timestamp", LongType(), True),
    ]
)
source_csv = f"{raw_folder_path}/ratings.csv"
target_delta = f"{bronze_folder_path}/ratings"

ratings_df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("mode", "PERMISSIVE")
        .schema(ratings_schema)
        .csv(source_csv)
    )
    
ratings_df = ratings_df \
    .withColumn("rated_at", to_timestamp(from_unixtime(col("timestamp")), 'yyyy-MM-dd HH:mm:ss')) \
    .withColumn("ingest_timestamp", current_timestamp()) \
    .drop("timestamp")
    
ratings_df.write.mode("overwrite").format("delta").save(target_delta)

display(ratings_df.limit(5))